# 🛍️ Retail Sales Analytics & Demand Forecasting
**End-to-End Data Science Project**

---
**Sections:**
1. Setup & Imports
2. Data Loading
3. Automated Data Cleaning
4. Exploratory Data Analysis (EDA)
5. KPI Computation
6. SQL Analytics
7. Machine Learning Forecasting
8. Report Generation


In [ ]:
# ── 1. SETUP ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
print('✅ All libraries imported successfully')

In [ ]:
# ── 2. DATA LOADING ───────────────────────────────────────────────────────────
# Change path to your dataset
DATA_PATH = '../data/your_dataset.csv'   # <-- update this

if DATA_PATH.endswith('.csv'):
    df_raw = pd.read_csv(DATA_PATH)
else:
    df_raw = pd.read_excel(DATA_PATH)

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# Dataset info
print('=== DATASET INFO ===')
df_raw.info()
print('\n=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])
print(f'\nDuplicates: {df_raw.duplicated().sum()}')

In [ ]:
# ── 3. DATA CLEANING ──────────────────────────────────────────────────────────
from utils.data_cleaner import full_clean_pipeline

df, cleaning_log = full_clean_pipeline(df_raw.copy())

print('=== CLEANING LOG ===')
for k, v in cleaning_log.items():
    print(f'\n{k}:')
    print(f'  {v}')

print(f'\nRows before: {len(df_raw):,}  →  after: {len(df):,}')

In [ ]:
# ── 4. EDA ────────────────────────────────────────────────────────────────────
# Descriptive statistics
print('=== DESCRIPTIVE STATISTICS ===')
df.describe(include='all').T

In [ ]:
# Correlation heatmap
num = df.select_dtypes(include=np.number)
if len(num.columns) >= 2:
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, ax=ax, linewidths=0.5)
    ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Numeric distributions
num_cols = num.columns.tolist()
fig, axes = plt.subplots(nrows=-(-len(num_cols)//3), ncols=3,
                          figsize=(16, 4*(-(-len(num_cols)//3))))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Numeric Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Configure column mapping (update to match your dataset) ──────────────────
from utils.data_loader import auto_detect_col_map
col_map = auto_detect_col_map(df)
print('Auto-detected column mapping:')
for k, v in col_map.items():
    print(f'  {k}: {v}')

In [ ]:
# Sales trend (if date and sales columns exist)
dc = col_map.get('date_col')
sc = col_map.get('sales_col')

if dc and sc:
    tmp = df[[dc, sc]].copy()
    tmp[dc] = pd.to_datetime(tmp[dc], errors='coerce')
    monthly = tmp.dropna(subset=[dc]).set_index(dc).resample('ME')[sc].sum().reset_index()

    fig = px.line(monthly, x=dc, y=sc, markers=True,
                  title='Monthly Sales Trend',
                  color_discrete_sequence=['#0ea5e9'])
    fig.update_layout(template='plotly_dark', height=400)
    fig.show()
else:
    print('Set date_col and sales_col in col_map to see trend chart.')

In [ ]:
# Category analysis
cc = col_map.get('category_col')
sc = col_map.get('sales_col')

if cc and sc:
    cat_rev = df.groupby(cc)[sc].sum().sort_values(ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    cat_rev.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title(f'Revenue by {cc}', fontweight='bold')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=45)

    axes[1].pie(cat_rev, labels=cat_rev.index, autopct='%1.1f%%',
                colors=plt.cm.tab10.colors)
    axes[1].set_title(f'Revenue Share by {cc}', fontweight='bold')

    plt.tight_layout()
    plt.savefig('../reports/category_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(cat_rev)
else:
    print('Set category_col and sales_col in col_map.')

In [ ]:
# ── 5. KPI COMPUTATION ────────────────────────────────────────────────────────
from utils.kpi import compute_kpis, generate_insights

kpis = compute_kpis(df, col_map)
print('=== KEY PERFORMANCE INDICATORS ===')
for k, v in kpis.items():
    if v is not None:
        print(f'  {k:25s}: {v}')

print('\n=== BUSINESS INSIGHTS ===')
insights = generate_insights(df, col_map, kpis)
for ins in insights:
    clean = ins.replace('**','')
    print(f'  {clean}')

In [ ]:
# ── 6. SQL ANALYTICS ──────────────────────────────────────────────────────────
from utils.sql_engine import build_sql_reports, run_query, save_to_db

save_to_db(df)
sql_reports = build_sql_reports(df, col_map)

for report_name, rdf in sql_reports.items():
    print(f'\n=== {report_name.upper()} ===')
    print(rdf.to_string(index=False))

In [ ]:
# Custom SQL
custom_query = "SELECT * FROM sales_data LIMIT 5"
run_query(custom_query)

In [ ]:
# ── 7. MACHINE LEARNING ───────────────────────────────────────────────────────
from utils.ml_models import run_forecasting_pipeline

target = col_map.get('sales_col')  # or change to any numeric column
date   = col_map.get('date_col')

if target:
    print(f'Training models to predict: {target}')
    results = run_forecasting_pipeline(df, target, date,
                                        save_dir='../models/')
    if 'error' in results:
        print('Error:', results['error'])
    else:
        print(f'\n✅ Best Model: {results["best_model"]}')
        for model in ['Linear Regression', 'Random Forest']:
            print(f'\n{model}:')
            for metric, val in results[model]['metrics'].items():
                print(f'  {metric}: {val}')
else:
    print('Set sales_col in col_map to run ML forecasting.')

In [ ]:
# Actual vs Predicted plot
if target and 'error' not in results:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, model_name in zip(axes, ['Linear Regression', 'Random Forest']):
        y_test = results['y_test'][:100]
        preds  = results[model_name]['predictions'][:100]
        ax.plot(y_test, label='Actual',    color='steelblue', linewidth=2)
        ax.plot(preds,  label='Predicted', color='orange', linewidth=2, linestyle='--')
        ax.set_title(model_name, fontweight='bold')
        ax.legend()
        m = results[model_name]['metrics']
        ax.set_xlabel(f"R²={m['R²']}  RMSE={m['RMSE']}")
    plt.suptitle('Actual vs Predicted', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Feature importance
if target and 'error' not in results:
    fi = results['Random Forest']['feature_importance']
    if not fi.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        fi.head(15).sort_values('Importance').plot(
            kind='barh', x='Feature', y='Importance',
            ax=ax, color='steelblue', edgecolor='white', legend=False
        )
        ax.set_title('Feature Importance — Random Forest', fontweight='bold')
        plt.tight_layout()
        plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# ── 8. REPORT GENERATION ──────────────────────────────────────────────────────
from utils.report_generator import generate_csv_summary, generate_excel_report, generate_pdf_report
import os

os.makedirs('../reports', exist_ok=True)

# CSV
csv_bytes = generate_csv_summary(df)
with open('../reports/stats_summary.csv', 'wb') as f:
    f.write(csv_bytes)
print('✅ CSV summary saved to reports/stats_summary.csv')

# Excel
xl_bytes = generate_excel_report(df, kpis, col_map, sql_reports)
with open('../reports/retail_analytics.xlsx', 'wb') as f:
    f.write(xl_bytes)
print('✅ Excel report saved to reports/retail_analytics.xlsx')

# PDF
pdf_bytes = generate_pdf_report(df, kpis, insights, cleaning_log)
with open('../reports/retail_analytics_report.pdf', 'wb') as f:
    f.write(pdf_bytes)
print('✅ PDF report saved to reports/retail_analytics_report.pdf')

---
## ✅ Analysis Complete!

All reports saved to the `reports/` folder. Run `streamlit run ../app.py` for the interactive dashboard.